# <center>**02 - Benchmark Epoch**<center>

Notebook chạy thử một số batch cho từng kiến trúc, từ đó ước lượng thời gian cần thiết cho một epoch huấn luyện và đánh giá. Đây chỉ là bước benchmark; không lưu checkpoint hoặc kết quả huấn luyện.

In [ ]:
import logging
import sys
import time
from pathlib import Path

import torch
import torch.nn as nn
from torch.utils.data import DataLoader

ROOT = next(
    p
    for p in [Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent]
    if (p / "requirements.txt").exists()
)
sys.path.insert(0, str(ROOT / "src"))

from models import MODEL_NAMES, build_model, count_params
from train_utils import (
    FERDataset,
    get_device,
    load_processed,
    run_epoch,
    set_seed,
)

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(message)s",
)
log = logging.getLogger("bench")

BATCH_SIZE = 64
N_BATCHES = 30  # Số batch dùng để đo; bỏ batch đầu tiên vì thời gian khởi động.

## **1. Chuẩn bị dữ liệu**

Đặt seed để kết quả có thể tái lập, chọn thiết bị chạy và nạp dữ liệu đã xử lý. Tạo DataLoader cho tập Training (có augmentation) và Validation (không augmentation).


In [2]:
set_seed(42)
device = get_device()
splits, stats = load_processed(ROOT / "data" / "processed")

x_tr, y_tr = splits["train"]
x_va, y_va = splits["valid"]

train_dataset = FERDataset(
    x_tr,
    y_tr,
    stats["mean"],
    stats["std"],
    augment=True,
)
valid_dataset = FERDataset(
    x_va,
    y_va,
    stats["mean"],
    stats["std"],
)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0,
)
valid_loader = DataLoader(
    valid_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
)

log.info(
    "Số batch: train=%d, valid=%d",
    len(train_loader),
    len(valid_loader),
)

2026-11-23 15:15:17,817 | INFO | Thiết bị: cpu
2026-11-23 15:15:18,345 | INFO | Số batch: train=449, valid=57


## **2. Đo từng kiến trúc**

Chạy thử một batch để giảm ảnh hưởng của thời gian khởi tạo. Sau đó, đo một số batch trên Training và Validation rồi ngoại suy thời gian cho toàn bộ hai tập.


In [3]:
results = {}

for name in MODEL_NAMES:
    model = build_model(name).to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

    # Khởi động model; không tính batch này vào thời gian benchmark.
    run_epoch(
        model,
        train_loader,
        device,
        criterion,
        optimizer,
        max_batches=1,
    )

    start_time = time.perf_counter()
    _, _, train_batches = run_epoch(
        model,
        train_loader,
        device,
        criterion,
        optimizer,
        max_batches=N_BATCHES,
    )
    train_seconds = (
        (time.perf_counter() - start_time)
        / train_batches
        * len(train_loader)
    )

    start_time = time.perf_counter()
    _, _, valid_batches = run_epoch(
        model,
        valid_loader,
        device,
        criterion,
        max_batches=N_BATCHES,
    )
    valid_seconds = (
        (time.perf_counter() - start_time)
        / valid_batches
        * len(valid_loader)
    )

    epoch_seconds = train_seconds + valid_seconds
    results[name] = epoch_seconds

    log.info(
        "%-10s | params %10s | ước lượng train %.0f s + "
        "valid %.0f s = %.1f phút/epoch",
        name,
        f"{count_params(model):,}",
        train_seconds,
        valid_seconds,
        epoch_seconds / 60,
    )

2026-11-23 15:17:33,242 | INFO | small      | params 38,901,191 | ước lượng train 1513 s + valid 57 s = 26.2 phút/epoch
2026-11-23 15:19:12,205 | INFO | medium     | params 11,327,303 | ước lượng train 1094 s + valid 44 s = 19.0 phút/epoch
2026-11-23 15:21:27,324 | INFO | medium_01  | params 12,101,895 | ước lượng train 1497 s + valid 60 s = 25.9 phút/epoch
2026-11-23 15:24:49,856 | INFO | large      | params 20,406,599 | ước lượng train 2239 s + valid 90 s = 38.8 phút/epoch


## **3. Ước lượng số epoch**

Nhập ngân sách thời gian (phút) dành cho mỗi kiến trúc. Số epoch khả thi được ước lượng bằng cách chia ngân sách này cho thời gian benchmark của một epoch.


In [4]:
BUDGET_MIN = 60  # Ví dụ: mỗi kiến trúc được tối đa 60 phút.

for name, epoch_seconds in results.items():
    estimated_epochs = int(BUDGET_MIN * 60 // epoch_seconds)
    log.info(
        "%-10s | trong %d phút: khoảng %d epoch",
        name,
        BUDGET_MIN,
        estimated_epochs,
    )

2026-11-23 15:24:49,873 | INFO | small      | trong 60 phút: khoảng 2 epoch
2026-11-23 15:24:49,874 | INFO | medium     | trong 60 phút: khoảng 3 epoch
2026-11-23 15:24:49,875 | INFO | medium_01  | trong 60 phút: khoảng 2 epoch
2026-11-23 15:24:49,876 | INFO | large      | trong 60 phút: khoảng 1 epoch
